In [31]:
import pandas as pd
from pathlib import Path   
import numpy as np

In [32]:
result_path = Path("../results/downstream_task")
bias_types = ["less_positive_class"]
metrics = ["AUROC"]
less_bias_strengths = ["0.1"]
method_name_replacer = {
                        "uniform": "Uniform", 
                        "kmm": "KMM", 
                        "psa": "PSA", 
                        "mrs-forest": "MRS", 
                        "fw-mrs-temperature": "FW-MRS",
                        "fw-mrs-temperature-svm": "FW-MRS$_{SVM}$",
                        }
data_set_replacer = {
                    "folktables_employment": "Employment", 
                    "folktables_income": "Income",
                    "breast_cancer": "Breast Cancer", 
                    "hr_analytics": "HR Analytic", 
                    "loan_prediction": "Loan",
                    "diabetes": "Diabetes", 
                    "german_credit": "German Credit", 
                    "bank_marketing": "Bank Marketing"
                    }

In [33]:
aurocs = []
auprcs = []
dict_list = []
for dataset in data_set_replacer.keys():
    for bias_type in bias_types:
        for method in method_name_replacer.keys():
            for bias_strength in less_bias_strengths:
                json_file = result_path / dataset / bias_type /  bias_strength/ method / "classification_results.json"
                try:
                    result_file = pd.read_json(str(json_file))
                except FileNotFoundError:
                    continue
                dict_list.append(
                    {
                        "Method": method, "Data Set": dataset, 
                        "AUROC Mean": result_file["random forest auroc"]["mean"], 
                        "AUROC Std": result_file["random forest auroc"]["sd"], 
                        "Bias Type": bias_type, "Bias Strength": bias_strength,
                        "Dropped Samples Mean": result_file["dropped_samples"]["mean"],
                        "Dropped Samples Std": result_file["dropped_samples"]["std"],
                        "SVM PAD Mean": result_file["svm pad"]["mean"],
                        "SVM PAD Std": result_file["svm pad"]["sd"],
                        "RF Domain AUROC Mean": result_file["rf domain auroc"]["mean"],
                        "RF Domain AUROC Std": result_file["rf domain auroc"]["sd"],
                    }
                                )
result_df = pd.DataFrame(data=dict_list)

In [34]:
result_df = result_df.replace(method_name_replacer)
result_df

,Method,Data Set,AUROC Mean,AUROC Std,Bias Type,Bias Strength,Dropped Samples Mean,Dropped Samples Std,SVM PAD Mean,SVM PAD Std,RF Domain AUROC Mean,RF Domain AUROC Std
0,Uniform,folktables_employment,0.870601,0.010491,less_positive_class,0.1,0.000000,0.000000,0.357066,0.036606,0.630367,0.010368
1,KMM,folktables_employment,0.856631,0.013580,less_positive_class,0.1,0.000000,0.000000,0.099177,0.023438,0.525253,0.008685
2,PSA,folktables_employment,0.867401,0.010937,less_positive_class,0.1,0.040000,0.280000,0.042419,0.011849,0.528106,0.009637
3,MRS,folktables_employment,0.869954,0.009992,less_positive_class,0.1,203.900000,35.061232,0.289446,0.041109,0.602020,0.012732
4,FW-MRS,folktables_employment,0.866548,0.011815,less_positive_class,0.1,195.833333,36.854228,0.295478,0.038170,0.604088,0.011630
5,FW-MRS$_{SVM}$,folktables_employment,0.862458,0.011854,less_positive_class,0.1,269.600000,32.616560,0.148346,0.039321,0.576571,0.012588
6,Uniform,folktables_income,0.838084,0.012878,less_positive_class,0.1,0.000000,0.000000,0.352737,0.041463,0.612679,0.014196
7,KMM,folktables_income,0.820208,0.014266,less_positive_class,0.1,0.000000,0.000000,0.092928,0.026392,0.527638,0.007296
8,PSA,folktables_income,0.831057,0.013985,less_positive_class,0.1,0.160000,0.703136,0.044959,0.014487,0.519376,0.007730
9,MRS,folktables_income,0.837651,0.013005,less_positive_class,0.1,199.400000,49.138987,0.285773,0.052469,0.587054,0.018934


In [35]:
for bias_type in bias_types:
    for bias_strength in less_bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for dataset in data_set_replacer.keys():
            mean_auroc_values = []
            std_auroc_values = []
            for method in result_df["Method"].unique():
                try:
                    mean_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["AUROC Mean"].iloc[0]
                    mean_auroc_values.append(np.round(mean_auroc, 3))

                    std_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["AUROC Std"].iloc[0]
                    std_auroc_values.append(np.round(std_auroc, 2))
                except IndexError:
                    mean_auroc_values.append(0)
                    std_auroc_values.append(0)

            print(f"{data_set_replacer[dataset]} \
& ${mean_auroc_values[0]}\\pm{std_auroc_values[0]}$ \
& ${mean_auroc_values[1]}\\pm{std_auroc_values[1]}$ \
& ${mean_auroc_values[2]}\\pm{std_auroc_values[2]}$ \
& ${mean_auroc_values[3]}\\pm{std_auroc_values[3]}$ \
& ${mean_auroc_values[4]}\\pm{std_auroc_values[4]}$ \
& ${mean_auroc_values[5]}\\pm{std_auroc_values[5]}$ \
\\\\")
        print("\n")

less_positive_class, 0.1
Employment & $0.871\pm0.01$ & $0.857\pm0.01$ & $0.867\pm0.01$ & $0.87\pm0.01$ & $0.867\pm0.01$ & $0.862\pm0.01$ \\
Income & $0.838\pm0.01$ & $0.82\pm0.01$ & $0.831\pm0.01$ & $0.838\pm0.01$ & $0.834\pm0.01$ & $0.832\pm0.01$ \\
Breast Cancer & $0.988\pm0.01$ & $0.989\pm0.01$ & $0.988\pm0.01$ & $0.989\pm0.01$ & $0.98\pm0.01$ & $0.976\pm0.01$ \\
HR Analytic & $0.753\pm0.02$ & $0.749\pm0.02$ & $0.75\pm0.02$ & $0.751\pm0.02$ & $0.751\pm0.02$ & $0.752\pm0.02$ \\
Loan & $0.658\pm0.08$ & $0.61\pm0.1$ & $0.628\pm0.1$ & $0.638\pm0.1$ & $0.612\pm0.09$ & $0.574\pm0.1$ \\
Diabetes & $0.791\pm0.02$ & $0.781\pm0.02$ & $0.787\pm0.02$ & $0.789\pm0.02$ & $0.786\pm0.02$ & $0.783\pm0.03$ \\
German Credit & $0.667\pm0.05$ & $0.649\pm0.05$ & $0.659\pm0.06$ & $0.668\pm0.05$ & $0.644\pm0.06$ & $0.668\pm0.04$ \\
Bank Marketing & $0.847\pm0.02$ & $0.832\pm0.03$ & $0.839\pm0.03$ & $0.846\pm0.02$ & $0.845\pm0.02$ & $0.839\pm0.02$ \\




In [36]:
for bias_type in bias_types:
    for bias_strength in less_bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for dataset in data_set_replacer.keys():
            mean_pad_values = []
            std_pad_values = []
            for method in result_df["Method"].unique():
                try:
                    mean_pad = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["SVM PAD Mean"].iloc[0]
                    mean_pad_values.append(np.round(mean_pad, 3))

                    std_pad = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["SVM PAD Std"].iloc[0]
                    std_pad_values.append(np.round(std_pad, 2))
                except IndexError:
                    mean_pad_values.append(0)
                    std_pad_values.append(0)

            print(f"{data_set_replacer[dataset]} \
& ${mean_pad_values[0]}\\pm{std_pad_values[0]}$ \
& ${mean_pad_values[1]}\\pm{std_pad_values[1]}$ \
& ${mean_pad_values[2]}\\pm{std_pad_values[2]}$ \
& ${mean_pad_values[3]}\\pm{std_pad_values[3]}$ \
& ${mean_pad_values[4]}\\pm{std_pad_values[4]}$ \
& ${mean_pad_values[5]}\\pm{std_pad_values[5]}$ \
& \\\\")
        print("\n")

less_positive_class, 0.1
Employment & $0.357\pm0.04$ & $0.099\pm0.02$ & $0.042\pm0.01$ & $0.289\pm0.04$ & $0.295\pm0.04$ & $0.148\pm0.04$ & \\
Income & $0.353\pm0.04$ & $0.093\pm0.03$ & $0.045\pm0.01$ & $0.286\pm0.05$ & $0.267\pm0.06$ & $0.12\pm0.04$ & \\
Breast Cancer & $0.934\pm0.05$ & $0.142\pm0.05$ & $0.37\pm0.09$ & $0.668\pm0.09$ & $0.513\pm0.15$ & $0.366\pm0.13$ & \\
HR Analytic & $0.104\pm0.03$ & $0.036\pm0.01$ & $0.035\pm0.01$ & $0.086\pm0.03$ & $0.086\pm0.03$ & $0.079\pm0.03$ & \\
Loan & $0.35\pm0.11$ & $0.246\pm0.06$ & $0.172\pm0.05$ & $0.281\pm0.1$ & $0.244\pm0.11$ & $0.206\pm0.09$ & \\
Diabetes & $0.106\pm0.03$ & $0.044\pm0.01$ & $0.037\pm0.01$ & $0.095\pm0.04$ & $0.09\pm0.03$ & $0.08\pm0.03$ & \\
German Credit & $0.158\pm0.07$ & $0.116\pm0.04$ & $0.09\pm0.03$ & $0.141\pm0.06$ & $0.122\pm0.06$ & $0.077\pm0.0$ & \\
Bank Marketing & $0.08\pm0.03$ & $0.063\pm0.02$ & $0.037\pm0.01$ & $0.065\pm0.02$ & $0.07\pm0.03$ & $0.064\pm0.02$ & \\




In [37]:
for bias_type in bias_types:
    for bias_strength in less_bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for dataset in data_set_replacer.keys():
            mean_domain_values = []
            std_domain_values = []
            for method in result_df["Method"].unique():
                try:
                    mean_domain = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["RF Domain AUROC Mean"].iloc[0]
                    mean_domain_values.append(np.round(mean_domain, 3))

                    std_domain = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["RF Domain AUROC Std"].iloc[0]
                    std_domain_values.append(np.round(std_domain, 2))
                except IndexError:
                    mean_domain_values.append(0)
                    std_domain_values.append(0)

            print(f"{data_set_replacer[dataset]} \
& ${mean_domain_values[0]}\\pm{std_domain_values[0]}$ \
& ${mean_domain_values[1]}\\pm{std_domain_values[1]}$ \
& ${mean_domain_values[2]}\\pm{std_domain_values[2]}$ \
& ${mean_domain_values[3]}\\pm{std_domain_values[3]}$ \
& ${mean_domain_values[4]}\\pm{std_domain_values[4]}$ \
& ${mean_domain_values[5]}\\pm{std_domain_values[5]}$ \
& \\\\")
        print("\n")

less_positive_class, 0.1
Employment & $0.63\pm0.01$ & $0.525\pm0.01$ & $0.528\pm0.01$ & $0.602\pm0.01$ & $0.604\pm0.01$ & $0.577\pm0.01$ & \\
Income & $0.613\pm0.01$ & $0.528\pm0.01$ & $0.519\pm0.01$ & $0.587\pm0.02$ & $0.584\pm0.02$ & $0.572\pm0.02$ & \\
Breast Cancer & $0.731\pm0.02$ & $0.592\pm0.03$ & $0.583\pm0.03$ & $0.649\pm0.03$ & $0.658\pm0.03$ & $0.678\pm0.05$ & \\
HR Analytic & $0.533\pm0.01$ & $0.514\pm0.01$ & $0.511\pm0.0$ & $0.527\pm0.01$ & $0.525\pm0.01$ & $0.526\pm0.01$ & \\
Loan & $0.57\pm0.03$ & $0.585\pm0.02$ & $0.554\pm0.02$ & $0.551\pm0.02$ & $0.553\pm0.02$ & $0.55\pm0.02$ & \\
Diabetes & $0.525\pm0.01$ & $0.522\pm0.01$ & $0.512\pm0.0$ & $0.522\pm0.01$ & $0.52\pm0.01$ & $0.519\pm0.01$ & \\
German Credit & $0.538\pm0.02$ & $0.547\pm0.02$ & $0.533\pm0.01$ & $0.534\pm0.01$ & $0.531\pm0.01$ & $0.523\pm0.01$ & \\
Bank Marketing & $0.523\pm0.01$ & $0.521\pm0.01$ & $0.512\pm0.0$ & $0.517\pm0.01$ & $0.518\pm0.01$ & $0.517\pm0.01$ & \\




In [38]:
result_df.round(3)

,Method,Data Set,AUROC Mean,AUROC Std,Bias Type,Bias Strength,Dropped Samples Mean,Dropped Samples Std,SVM PAD Mean,SVM PAD Std,RF Domain AUROC Mean,RF Domain AUROC Std
0,Uniform,folktables_employment,0.871,0.010,less_positive_class,0.1,0.000,0.000,0.357,0.037,0.630,0.010
1,KMM,folktables_employment,0.857,0.014,less_positive_class,0.1,0.000,0.000,0.099,0.023,0.525,0.009
2,PSA,folktables_employment,0.867,0.011,less_positive_class,0.1,0.040,0.280,0.042,0.012,0.528,0.010
3,MRS,folktables_employment,0.870,0.010,less_positive_class,0.1,203.900,35.061,0.289,0.041,0.602,0.013
4,FW-MRS,folktables_employment,0.867,0.012,less_positive_class,0.1,195.833,36.854,0.295,0.038,0.604,0.012
5,FW-MRS$_{SVM}$,folktables_employment,0.862,0.012,less_positive_class,0.1,269.600,32.617,0.148,0.039,0.577,0.013
6,Uniform,folktables_income,0.838,0.013,less_positive_class,0.1,0.000,0.000,0.353,0.041,0.613,0.014
7,KMM,folktables_income,0.820,0.014,less_positive_class,0.1,0.000,0.000,0.093,0.026,0.528,0.007
8,PSA,folktables_income,0.831,0.014,less_positive_class,0.1,0.160,0.703,0.045,0.014,0.519,0.008
9,MRS,folktables_income,0.838,0.013,less_positive_class,0.1,199.400,49.139,0.286,0.052,0.587,0.019


In [42]:
result_df["Rank AUROC"] = result_df.round(3).groupby("Data Set")["AUROC Mean"].rank(ascending=False)
result_df["Rank PAD"] = result_df.round(3).groupby("Data Set")["SVM PAD Mean"].rank(ascending=True)
result_df["Rank Domain"] = result_df.round(3).groupby("Data Set")["RF Domain AUROC Mean"].rank(ascending=True)
result_df[["Method", "Rank AUROC", "Rank PAD", "Rank Domain"]].groupby("Method").mean()

,Rank AUROC,Rank PAD,Rank Domain
Method,,,
FW-MRS,4.0000,4.1875,3.5000
FW-MRS$_{SVM}$,4.2500,2.5000,2.6875
KMM,5.1875,2.2500,3.5625
MRS,2.0000,4.6875,3.7500
PSA,3.9375,1.3750,1.7500
Uniform,1.6250,6.0000,5.7500
